# 1단계) PDF 로딩 & 텍스트 추출 (PyMuPDFLoader)

In [ ]:
# STEP 1: PDF 로딩 & 텍스트 추출 검증
import os
from glob import glob
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader  # pip install pymupdf 필수

# === 사용자 설정 ===
PDF_DIR = "file/도로 교통법"   # PDF들이 들어있는 폴더 경로로 바꾸세요 (예: "./data_pdf")
MAX_FILES = 5          # 너무 많으면 처음엔 몇 개만 로딩해서 테스트

# === 로드 ===
pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
if not pdf_paths:
    raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다. 경로를 확인하세요.")

pdf_paths = pdf_paths[:MAX_FILES]  # 처음엔 일부만 테스트
print(f"[INFO] 대상 PDF 개수: {len(pdf_paths)}")

all_docs = []  # LangChain Document 객체 리스트 (페이지 단위)
for p in tqdm(pdf_paths, desc="Loading PDFs"):
    loader = PyMuPDFLoader(p)
    docs = loader.load()  # 페이지별 Document 리스트
    all_docs.extend(docs)

print(f"[INFO] 로드된 페이지 수: {len(all_docs)}")

# === 간단 검증: 앞 몇 페이지 메타/내용 스니펫 출력 ===
for i, d in enumerate(all_docs[:3], start=1):
    print(f"\n--- 샘플 페이지 {i} ---")
    print("source:", d.metadata.get("source"))
    print("page:", d.metadata.get("page"))
    text = d.page_content.strip().replace("\n", " ")
    print("text snippet:", (text[:300] + "...") if len(text) > 300 else text)

# 2단계) 전처리 유틸 + 적용 + 검증

In [ ]:
# ===========================
# 법령 PDF 전처리 + 조문 단위 스플리팅
# ===========================
import os, re
from glob import glob
from typing import List
from tqdm import tqdm
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"      # 모든 PDF가 들어있는 폴더 경로
MAX_FILES = None          # None이면 전체, 숫자 지정시 일부만 처리
PRINT_SAMPLE_DOCS = 5     # 조문 단위로 자른 뒤 샘플 몇 개 출력
SNIPPET_CHARS = 300       # 샘플 출력 글자수

# ---------- 제거 규칙 ----------
TOP_BRACKET_META_RE = re.compile(r"^\s*(?:\[[^\]\n]+\]\s*)+", flags=re.MULTILINE)

BASE_HEADER_PATTERNS = [
    r"^\s*법제처\s*.*국가법령정보센터.*$",
    r"^\s*국가법령정보센터.*$",
    r"^\s*www\.law\.go\.kr\s*$",
    r"^\s*페이지\s*\d+\s*$",
    r"^\s*\d+\s*$",
    r"^\s*[-–—―]+\s*$",
]
PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
AMENDMENT_LINE_RE = re.compile(
    r"^\s*\[(?:[^]\n]*?(개정|신설|삭제|전문개정)[^]\n]*?)\]\s*$"
)
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")

HEADER_LINE_RE = re.compile("|".join(BASE_HEADER_PATTERNS), re.IGNORECASE)

def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    name = TITLE_DATE_SUFFIX_RE.sub("", name).strip()
    return name

def is_header_like_line(line: str) -> bool:
    if HEADER_LINE_RE.search(line):
        return True
    if PHONE_RE.search(line) and ORG_HINT_RE.search(line):
        return True
    if AMENDMENT_LINE_RE.search(line):
        return True
    return False

def clean_text_legal(text: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")
    text = TOP_BRACKET_META_RE.sub("", text, count=1)

    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_header_like_line(line):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"([^\n])\n([^\n])", r"\1 \2", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)
    return txt.strip()

# ---------- 조문 단위 스플리팅 ----------
ARTICLE_HEAD = re.compile(r"(제\s*\d+조(\s*의\s*\d+)?[^\n]*)")

def split_into_articles(text: str, law_title: str, source_meta: dict) -> List[Document]:
    """제N조 기준으로 분할하고, 각 조문 앞에 law_title 삽입"""
    matches = list(ARTICLE_HEAD.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        chunk = f"{law_title} {chunk}"  # law_title 삽입
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 실행 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        docs = loader.load()  # page-level
        law_title = normalize_title_from_source(p)

        # 페이지 전처리 후 병합
        page_texts = [clean_text_legal(d.page_content or "") for d in docs]
        merged_text = "\n\n".join([t for t in page_texts if t.strip()])

        # 조문 단위 스플리팅
        articles = split_into_articles(merged_text, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(articles)

    print(f"[INFO] 전체 PDF 처리 완료. 총 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content[:SNIPPET_CHARS])

### 
- [x] 조항 단위로 나누기
- [x] <전문 개설> 등등 삭제
- [x] 제 n장 삭제
- [x] 맨 위 도로교통법 시행규칙 삭제


In [ ]:
# ===========================
# 법령 PDF 전처리 + "조" 단위 스플리팅 (조의 n 은 같은 조에 유지)
# ===========================
import os
import re
from glob import glob
from typing import List
from tqdm import tqdm

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document

# ---------- 사용자 설정 ----------
PDF_DIR = "file/도로 교통법"   # 처리할 PDF 폴더
MAX_FILES = None               # None=전체, 숫자=일부
PRINT_SAMPLE_DOCS = 15         # 샘플 개수
SNIPPET_CHARS = 400            # 샘플 글자수

# ---------- 파일명 끝 (YYYYMMDD) 제거 ----------
TITLE_DATE_SUFFIX_RE = re.compile(r"\(\d{8}\)$")
def normalize_title_from_source(src: str) -> str:
    base = os.path.basename(src)
    name, _ = os.path.splitext(base)
    return TITLE_DATE_SUFFIX_RE.sub("", name).strip()

# ---------- 제거/정규화 규칙 ----------
TOP_BRACKET_BLOCK_RE = re.compile(r"^\s*(?:\[[^\]\n]+\]\s*){1,20}", flags=re.MULTILINE)

HEADER_FOOTER_RE = re.compile(
    "|".join([
        r"^\s*법제처.*국가법령정보센터.*$",
        r"^\s*국가법령정보센터.*$",
        r"^\s*www\.law\.go\.kr\s*$",
        r"^\s*페이지\s*\d+\s*$",
        r"^\s*[-–—―]+\s*$",
        r"^\s*\d+\s*$",
    ]),
    flags=re.IGNORECASE | re.MULTILINE,
)

PHONE_RE = re.compile(r"\b\d{2,3}-\d{3,4}-\d{4}\b")
ORG_HINT_RE = re.compile(r"(경찰청|행정안전부|법제처|담당부서|문의|전화|Fax|팩스|연락처)", re.IGNORECASE)
def is_contact_line(line: str) -> bool:
    return bool(PHONE_RE.search(line) and ORG_HINT_RE.search(line))

# 각/대괄호 메타(어디에 있어도 제거) — 본문 내용은 남기고 날짜/개정 표지만 제거
ANGLE_META_ANY_RE = re.compile(
    r'\s*<(?=[^>]*(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행))[^>]*>\s*'
)
BRACKET_META_ANY_RE = re.compile(
    r'\s*\[(?=[^\]]*(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행))[^\]]*\]\s*'
)
BROKEN_BRACKET_META_RE = re.compile(
    r'\[\s*(?=[^\]\n]*(개정|전문개정|전부개정|일부개정|신설|본조신설|삭제|정정|준용|부칙|제목개정|이동|전단|후단|종전|현행))[^\]\n]*'
)

# ❌ (중요) 조 헤더 자체를 지우는 규칙은 사용하지 않음
# 예: "제14조의3 삭제" 같은 헤더는 유지해야 함

# 문서 제목 라인 제거 (제목 문자열과 정확/유사 일치)
def is_title_line(line: str, law_title: str) -> bool:
    l = re.sub(r'\s+', '', line)
    t = re.sub(r'\s+', '', law_title)
    return l == t or l.endswith(t) or t.endswith(l)

# ---------- 본문 클리너 ----------
def clean_text_legal(text: str, law_title: str) -> str:
    text = text.replace("\u200b", "").replace("\ufeff", "")

    # 상단 메타/헤더푸터 제거
    text = TOP_BRACKET_BLOCK_RE.sub("", text, count=1)
    text = HEADER_FOOTER_RE.sub("", text)

    # 줄 단위 필터링
    kept = []
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            kept.append("")
            continue
        if is_contact_line(line):
            continue
        if is_title_line(line, law_title):
            continue
        kept.append(line)
    txt = "\n".join(kept)

    # 인라인 메타 제거 ([개정], <개정> 등)
    txt = ANGLE_META_ANY_RE.sub(" ", txt)
    txt = BRACKET_META_ANY_RE.sub(" ", txt)
    txt = BROKEN_BRACKET_META_RE.sub(" ", txt)

    # "문서제목 + 제N조" → 제목 제거 + 조 헤더 앞에 개행
    title_inline_re = re.compile(
        rf'(?m)(^|\n)\s*{re.escape(law_title)}\s+(?=제\s*\d+\s*조\b)'
    )
    txt = title_inline_re.sub(r"\n\n", txt)

    # 조 헤더 앞에 항상 2개 개행 보장 (조의n은 스플릿 대상이 아님)
    # 1) 문장 중간에 붙은 경우 줄 바꾸기
    txt = re.sub(r'(?!^)(?<!\n)(제\s*\d+\s*조\b)', r'\n\n\1', txt)
    # 2) 라인 시작도 최소 두 줄 개행
    txt = re.sub(r'(?m)(^)(\s*제\s*\d+\s*조\b)', r'\n\n\2', txt)

    # 하이픈 강제 줄바꿈 제거 + 개행 정리
    txt = re.sub(r"-\n", "", txt)
    txt = re.sub(r"[ \t]+", " ", txt)
    txt = re.sub(r"\n{3,}", "\n\n", txt)

    return txt.strip()

# ---------- 스플리팅 (오직 "제 N 조"만) ----------
ARTICLE_HEAD_ONLY_JO = re.compile(r"(?m)^\s*(제\s*\d+\s*조\b[^\n]*)")

def split_into_articles_only_jo(text: str, law_title: str, source_meta: dict) -> List[Document]:
    matches = list(ARTICLE_HEAD_ONLY_JO.finditer(text))
    if not matches:
        return [Document(page_content=f"{law_title}\n{text}", metadata=source_meta)]

    articles: List[Document] = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i+1].start() if i+1 < len(matches) else len(text)
        chunk = text[start:end].strip()
        if not chunk:
            continue
        # 조의n(예: 제2조의10) 같은 하위 조항은 이 블록 내부에 그대로 남음
        chunk = f"{law_title} {chunk}"
        articles.append(Document(page_content=chunk, metadata=source_meta))
    return articles

# ---------- 메인 ----------
if __name__ == "__main__":
    pdf_paths = sorted(glob(os.path.join(PDF_DIR, "*.pdf")))
    if MAX_FILES:
        pdf_paths = pdf_paths[:MAX_FILES]
    if not pdf_paths:
        raise FileNotFoundError(f"'{PDF_DIR}' 폴더에 PDF가 없습니다.")

    all_articles: List[Document] = []

    for p in tqdm(pdf_paths, desc="Processing PDFs"):
        loader = PyMuPDFLoader(p)
        pages = loader.load()

        law_title = normalize_title_from_source(p)
        page_texts = [clean_text_legal(d.page_content or "", law_title) for d in pages]
        merged = "\n\n".join([t for t in page_texts if t.strip()])

        docs = split_into_articles_only_jo(merged, law_title, {"source": p, "law_title": law_title})
        all_articles.extend(docs)

    print(f"[INFO] 전처리/분할 완료. 조문 단위 문서 수: {len(all_articles)}")

    # 샘플 출력
    for i, d in enumerate(all_articles[:PRINT_SAMPLE_DOCS], start=1):
        print(f"\n--- 조문 샘플 {i} ---")
        print("law_title:", d.metadata.get("law_title"))
        print(d.page_content)#[:SNIPPET_CHARS])

# 3단계) 임베딩(jina v3) → Chroma에 영구 저장 

In [ ]:
# ===========================
# 3단계: 임베딩(jina v3) + Chroma 영구 저장 (신규 API)
# ===========================
# 필요 패키지:
#   pip install sentence-transformers chromadb langchain_community tqdm

import os
import hashlib
from typing import List
from tqdm import tqdm

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma

# ---------- 사용자 설정 ----------
PERSIST_DIR = "chroma_store"         # 벡터스토어 저장 폴더
COLLECTION_NAME = "laws_by_article"  # 컬렉션명
DEVICE = "mps"                        # 안정 우선: "cpu" → 확인 후 "mps"/"cuda" 가능
BATCH_SIZE = 16                       # 임베딩 배치 크기 (메모리 상황에 맞게 조정)

# ---------- ① 폴더 통째 삭제 ----------
def reset_chroma_store(persist_dir: str):
    if os.path.exists(persist_dir):
        print(f"[INFO] 기존 Chroma 디렉토리 삭제: {persist_dir}")
        shutil.rmtree(persist_dir)
    os.makedirs(persist_dir, exist_ok=True)

# ---------- ① 임베딩 함수 (LangChain 호환) ----------
class SentenceTransformerEmbedding:
    def __init__(self, model_name: str = "jinaai/jina-embeddings-v3", device: str = "cpu"):
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True).tolist()
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

# ---------- ② ID 생성 유틸 (중복 방지) ----------
def doc_id(d: Document) -> str:
    """
    같은 조문을 여러 번 처리해도 동일 ID가 나오도록 결정적 해시 생성.
    (law_title + 첫 줄(제n조 라인) + source)
    """
    title = (d.metadata.get("law_title") or "").strip()
    source = (d.metadata.get("source") or "").strip()
    first_line = (d.page_content.splitlines()[0] if d.page_content else "").strip()
    key = f"{title}||{first_line}||{source}"
    return hashlib.sha256(key.encode("utf-8")).hexdigest()

# ---------- ③ 벡터스토어 생성/추가 (신규 Chroma 클라이언트 방식) ----------
def build_or_load_chroma(persist_dir: str, collection_name: str, embedding_fn) -> Chroma:
    """
    최신 Chroma 아키텍처:
      - PersistentClient(path=...) 사용
      - LangChain Chroma에 client=...로 주입
    """
    os.environ["CHROMA_TELEMETRY_ENABLED"] = "FALSE"  # 선택: 텔레메트리 비활성화
    os.makedirs(persist_dir, exist_ok=True)

    # ✅ 신규 방식: PersistentClient
    client = chromadb.PersistentClient(path=persist_dir)

    # LangChain의 Chroma에 client 주입 (persist_directory 넘겨도 되지만 client가 우선)
    vs = Chroma(
        client=client,
        collection_name=collection_name,
        embedding_function=embedding_fn,
    )
    return vs

def add_documents_in_batches(vs: Chroma, docs: List[Document], batch_size: int = 200):
    """
    LangChain의 add_documents는 중복 ID를 허용하므로,
    우리가 직접 IDs를 만들어 전달해서 중복을 피한다.
    """
    # 중복 제거 (ID 기준)
    unique = {}
    for d in docs:
        _id = doc_id(d)
        if _id not in unique:
            unique[_id] = d
    uniq_docs = list(unique.values())

    print(f"[INFO] 추가 대상 문서(중복 제거 후): {len(uniq_docs)}")

    for i in tqdm(range(0, len(uniq_docs), batch_size), desc="Adding docs"):
        batch = uniq_docs[i:i+batch_size]
        ids = [doc_id(d) for d in batch]
        texts = [d.page_content for d in batch]
        metadatas = [d.metadata for d in batch]
        vs.add_texts(texts=texts, metadatas=metadatas, ids=ids)

    # PersistentClient 사용 시에도 persist 호출은 무해/안전
    try:
        vs.persist()
    except Exception:
        pass

# ---------- ④ 실행 (이전 단계의 all_articles 사용) ----------
if __name__ == "__main__":
    # ⚠️ 이전 단계 코드에서 조 단위 문서 리스트를 만들었다고 가정:
    # from previous_step import all_articles
    # 여기서는 예시로 전역에 이미 존재한다고 가정한다.
    try:
        all_articles  # noqa: F821
    except NameError:
        raise RuntimeError("이전 단계의 'all_articles: List[Document]'가 필요합니다. "
                           "폴더 전처리+조문 스플릿 코드와 같은 런타임에서 실행하세요.")

    # 1) 임베딩 준비
    embedding_fn = SentenceTransformerEmbedding(device=DEVICE)

    # 2) 벡터스토어 로드/생성 (신규 API)
    vectorstore = build_or_load_chroma(PERSIST_DIR, COLLECTION_NAME, embedding_fn)

    # 3) 문서 추가
    add_documents_in_batches(vectorstore, all_articles, batch_size=200)

    # 4) 상태 확인
    try:
        count = vectorstore._collection.count()
        print(f"[OK] 저장 완료. 현재 컬렉션 문서 수: {count}")
    except Exception as e:
        print("[WARN] 카운트 확인 실패:", e)

# 4단계) 저장된 Chroma 로드 → 질의 (retriever)

In [ ]:
# ===========================
# 4단계: 저장된 Chroma 로드 & 질의
# ===========================
# 필요 패키지:
#   pip install sentence-transformers chromadb langchain_community

import os
import chromadb
from typing import List, Tuple
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma

# ---- 사용자 설정 ----
PERSIST_DIR = "chroma_store"          # 3단계에서 사용한 저장 폴더
COLLECTION_NAME = "laws_by_article"   # 3단계에서 사용한 컬렉션명
DEVICE = "cpu"                        # "cpu" → 안정 / "mps"(Apple) 또는 "cuda" 가능
TOP_K = 10                            # 기본 k

# ---- 임베딩 래퍼 (jina v3) ----
class SentenceTransformerEmbedding:
    def __init__(self, model_name: str = "jinaai/jina-embeddings-v3", device: str = "cpu"):
        self.model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self.model.encode(texts, batch_size=64, normalize_embeddings=True).tolist()
    def embed_query(self, text: str) -> List[float]:
        return self.model.encode([text], normalize_embeddings=True)[0].tolist()

# ---- 로드: PersistentClient + LangChain Chroma ----
embedding_fn = SentenceTransformerEmbedding(device=DEVICE)
client = chromadb.PersistentClient(path=PERSIST_DIR)
vectorstore = Chroma(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

# ---- 1) 가장 간단한 질의(retriever) ----
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
query = "어린이 보호구역의 제한 속도를 알려줘"
docs = retriever.invoke(query)

print(f"[RESULT - retriever] top {len(docs)} for: {query}\n")
for i, d in enumerate(docs, 1):
    print(f"[{i}] {d.metadata.get('law_title')} | source: {os.path.basename(d.metadata.get('source',''))}")
    print(d.page_content[:300].replace("\n", " ") + ("..." if len(d.page_content) > 300 else ""))
    print("-" * 80)

# ---- 2) 점수까지 보고 싶을 때 (similarity_search_with_score) ----
print("\n[RESULT - with scores]")
scored: List[Tuple] = vectorstore.similarity_search_with_score(query, k=TOP_K)
for i, (d, score) in enumerate(scored, 1):
    # 점수는 거리 기반(작을수록 유사). 코사인 유사도가 아닌 'distance' 값임에 유의.
    print(f"[{i}] score={score:.4f} | {d.metadata.get('law_title')} | {os.path.basename(d.metadata.get('source',''))}")
    print(d.page_content[:220].replace("\n", " ") + ("..." if len(d.page_content) > 220 else ""))
    print("-" * 80)

# ---- 3) 다양성 중시 검색(MMR) ----
retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": 50, "lambda_mult": 0.3},  # lambda_mult 낮을수록 다양성↑
)
docs_mmr = retriever_mmr.invoke("보행자 보호 의무는 무엇인가?")
print("\n[RESULT - MMR] 다양성 중시")
for i, d in enumerate(docs_mmr, 1):
    print(f"[{i}] {d.metadata.get('law_title')} | {os.path.basename(d.metadata.get('source',''))}")
    print(d.page_content[:240].replace("\n", " ") + ("..." if len(d.page_content) > 240 else ""))
    print("-" * 80)

# ---- 4) 특정 법령으로 필터링해서 검색(예: 도로교통법 시행규칙만) ----
# Chroma where 필터 사용: 정확 일치가 가장 안정적입니다. (부분 포함은 $contains 사용 가능)
FILTER_TITLE = "도로교통법 시행규칙(행정안전부령)(제00507호)"
filtered = vectorstore.similarity_search(
    "통학버스 관련 의무를 알려줘",
    k=TOP_K,
    filter={"law_title": {"$eq": FILTER_TITLE}},  # 또는 {"$contains": "도로교통법 시행규칙"}
)
print("\n[RESULT - filtered by law_title]")
for i, d in enumerate(filtered, 1):
    print(f"[{i}] {d.metadata.get('law_title')} | {os.path.basename(d.metadata.get('source',''))}")
    print(d.page_content[:240].replace("\n", " ") + ("..." if len(d.page_content) > 240 else ""))
    print("-" * 80)

In [ ]:
retriever.invoke("어린이 보호구역의 제한 속도를 알려줘")